# EDA — Cloud workload forecasting (`data-v1`)

**Where this analysis lives:** `notebooks/01_eda.ipynb` (narrative) and `notebooks/01_eda.py` (reproducible script). Figures are in `notebooks/figures/`.

## Anti-leakage protocol

- Panel integrity (row counts, cadence, join keys) may use the full release.
- Target dynamics, ACF, segments, peaks, and any statistic that could influence model choice use **only the first 110 observations** (`timestamp` 0–32700).
- The final 24 observations are **not** included in those summaries.

Run:

```bash
python notebooks/01_eda.py
```

In [ ]:
from pathlib import Path
import runpy

runpy.run_path(str(Path("01_eda.py").resolve() if Path("01_eda.py").exists() else Path("notebooks/01_eda.py")))

## Results (train, measured)

### Panel

- 10,000 VMs × 134 timestamps, 300 s cadence, no nulls and no duplicates.
- `active_vm_count(t) = 10000` at **every** timestamp. There is no churn in this sample (VMs were selected with complete traces).
- Forecasting `active_vm_count` is trivial; the signal is in `aggregate_cpu_utilization`.

### Aggregate series (110 points)

| Metric | Value |
|---|---:|
| Mean | 9.414% |
| Std | 0.194 |
| Min / max | 9.112 / 9.912 |
| In-sample persistence MAE (`y_t` vs `y_{t-1}`) | 0.121 |
| In-sample mean-baseline MAE | 0.150 |

The fleet mean moves by **tenths of a percentage point**. Persistence beats “predict the mean” inside train, consistent with ACF(1) ≈ 0.66.

![Fleet-mean CPU train](figures/train_aggregate_cpu.png)

Peaks (mean + 2σ ≈ 9.80) fall between t=16500 and t=20400 (~hours 4.6–5.7 of the trace). Elapsed hours 4–5 are highest; hours 7–9 decline. **There is no real calendar:** timestamps are seconds relative to the trace, not Azure time of day.

![ACF train](figures/train_acf.png)

| Lag | Time | ACF |
|---:|---:|---:|
| 1 | 5 min | 0.66 |
| 3 | 15 min | 0.66 |
| 12 | 60 min | 0.33 |
| 24 | 2 h | ≈ 0 |

Implication: short-term memory. There is **no** seasonality at the 2-hour horizon. A 24-step seasonal-naive is still a baseline worth implementing, but ACF does not support it.

### VM heterogeneity (train means)

![Histogram of per-VM mean CPU](figures/vm_mean_cpu_hist.png)

- VM median ≈ 5.5%; VM mean ≈ 9.4% (right tail).
- Top 1% of VMs ≈ 9.6% of fleet-mean CPU; top 10% ≈ 43.5%.
- 220 VMs with mean > 50%.
- Mean intra-interval spread (`max_cpu - min_cpu`) ≈ 12.3 points: 5-minute bursts that the fleet mean does not show.

![CPU by category](figures/cpu_by_category.png)

| vm_category | N | Mean | Median |
|---|---:|---:|---:|
| Interactive | 3091 | 11.51 | 5.58 |
| Unknown | 729 | 10.35 | 3.88 |
| Delay-insensitive | 6180 | 8.26 | 5.63 |

Interactive has a **heavier tail** (different means, similar medians). Core/memory buckets do not show a clean gradient; `>24` cores has n=10.

### Operational reading

Sizing from this subset’s **fleet mean** is almost a constant problem. Risk sits in hot VMs, intra-interval peaks, and the Interactive segment. An extra target (P95 or summed CPU) would be more actionable; the assignment still requires forecasting the mean.

Next task (when authorized): build the target series in pipeline code (`src/`) with the 110/24 split sealed, and evaluate temporal baselines **on train only**.